# Prepare Environment

В данном блокноте приведены эксперименты по сжатию модели. Для воспроизведения итогового варианта необходимо запустить код в следующих разделах:

1. Prepare Environment
2. Quantization
3. (Quantization) 4 bit - group size 128

Эксперименты проводились на платформе Kaggle с GPU T4x2

In [1]:
!pip install -q -U accelerate datasets hqq transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 9.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.3 MB/s eta 0:00:0000:010:01


## Load and Test Functions

In [2]:
import torch
import numpy as np
import random
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [4]:
MODEL_ID = 'Qwen/Qwen3-8B'
SUBSET_RATIO = 0.2
NBITS = 4
SAVE_PATH = f"qwen3-8b-hqq-{NBITS}bit"
ORIG_SIZE = 15622.631477355957
ORIG_METRIC = 0.7106

In [5]:
def load_mmlu(ratio=0.2):
    print(f"Loading MMLU subset ({ratio*100}%)...")
    dataset = load_dataset("cais/mmlu", "all", split="test")
    df = dataset.to_pandas()
    
    subset_df = df.groupby('subject', group_keys=False).apply(
        lambda x: x.sample(frac=ratio, random_state=42)
    )
    
    return subset_df

In [6]:
def evaluate_model(model, tokenizer, data_df):
    model.eval()
    choices = ['A', 'B', 'C', 'D']

    subjects = data_df['subject'].unique()
    subject_stats = {sub: {'correct': 0, 'total': 0} for sub in subjects}
    
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]
    prompt_template = "Question: {question}\nChoices:\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nAnswer:"

    with torch.no_grad():
        for _, row in tqdm(data_df.iterrows(), total=len(data_df), desc="Evaluating"):
            prompt = prompt_template.format(
                question=row['question'],
                a=row['choices'][0], b=row['choices'][1],
                c=row['choices'][2], d=row['choices'][3]
            )
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            outputs = model(**inputs)
            # Берем логиты последнего предсказанного токена
            last_logits = outputs.logits[0, -1, choice_ids]
            prediction = torch.argmax(last_logits).item()
            
            is_correct = (prediction == row['answer'])
            subject_stats[row['subject']]['total'] += 1
            if is_correct:
                subject_stats[row['subject']]['correct'] += 1

    results = []
    total_correct = 0
    total_samples = 0

    for sub, stats in subject_stats.items():
        acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
        results.append({'subject': sub, 'accuracy': acc, 'count': stats['total']})
        total_correct += stats['correct']
        total_samples += stats['total']
        
    df_results = pd.DataFrame(results)
    mean_accuracy = total_correct / total_samples
                
    return mean_accuracy, df_results

In [7]:
def get_size_mb(path):
    """Считает размер всех файлов весов в папке в Мегабайтах."""
    total_size = 0
    # Список расширений, которые относятся к весам
    weight_extensions = ('.safetensors', '.bin', '.pt', '.hqq_pack')
    
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024 * 1024)
    
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            if f.endswith(weight_extensions):
                fp = os.path.join(dirpath, f)
                total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

def calculate_lab_metrics(orig_metric, comp_metric, orig_path, comp_path, orig_size=None):
    # 1. Считаем размеры
    # Если оригинал в кэше HF, можно попробовать найти его там, 
    # либо просто замерить один раз и подставить число.
    if orig_size is None:
        orig_size = get_size_mb(orig_path)
    comp_size = get_size_mb(comp_path)
    
    # 2. Вычисляем Ratio
    compression_ratio = orig_size / comp_size
    
    # 3. Вычисляем Drop (по формуле из лабы)
    # Важно: если метрика вдруг выросла, drop должен быть 0
    performance_drop = max(0, (orig_metric - comp_metric) / orig_metric)
    
    # 4. Итоговый Score
    score = compression_ratio / (1 + performance_drop)
    
    return {
        "Original Size (MB)": orig_size,
        "Compressed Size (MB)": comp_size,
        "Ratio": compression_ratio,
        "Drop": performance_drop,
        "Final Score": score
    }

# Baseline

In [8]:
# Загрузка токенизатора и модели в FP16
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_baseline = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [9]:
# Считаем параметры
orig_params = sum(p.numel() for p in model_baseline.parameters())
print(f"Original parameters: {orig_params / 1e9:.2f}B")

Original parameters: 8.19B


In [10]:
model_baseline.save_pretrained('qwen3-baseline')
tokenizer.save_pretrained('qwen3-baseline')

('qwen3-baseline/tokenizer_config.json',
 'qwen3-baseline/special_tokens_map.json',
 'qwen3-baseline/chat_template.jinja',
 'qwen3-baseline/vocab.json',
 'qwen3-baseline/merges.txt',
 'qwen3-baseline/added_tokens.json',
 'qwen3-baseline/tokenizer.json')

In [11]:
get_size_mb('qwen3-baseline')

15622.631477355957

In [10]:
test_df = load_mmlu(SUBSET_RATIO)
test_df

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(


,question,subject,choices,answer
83,Statement 1 | If a group has an element of ord...,abstract_algebra,"[True, True, False, False, True, False, False,...",2
53,"Statement 1 | If G, H and K are groups of orde...",abstract_algebra,"[True, True, False, False, True, False, False,...",2
70,"(Z,*) is a group with a*b = a+b+1 for all a, b...",abstract_algebra,"[0, -2, a-2, (2+a)*-1]",3
45,"Statement 1 | For any two groups G and G', the...",abstract_algebra,"[True, True, False, False, True, False, False,...",2
44,"Let A and B be sets, f: A -> B and g: B -> A b...",abstract_algebra,"[True, True, False, False, True, False, False,...",2
...,...,...,...,...
13984,"According to the Korean foundation myth, who ...",world_religions,"[Hwanin, Hwanung, Joseon, Yi]",1
13902,"After the Bar Kochba revolt, where were the t...",world_religions,"[Palestine and Babylonia, Babylonia and Europe...",0
13961,What does the Tripitaka mean?,world_religions,"[Three gems, Three baskets, Three bodhisattvas...",1
14003,When was the State of Israel established?,world_religions,"[1947, 1948, 1945, 1949]",1


In [11]:
orig_metric, detailed_metric = evaluate_model(model_baseline, tokenizer, test_df)
print(f"Original Metric (MMLU subset): {orig_metric:.4f}")
detailed_metric

Evaluating: 100%|██████████| 2809/2809 [07:19<00:00,  6.39it/s]

Original Metric (MMLU subset): 0.7106


,subject,accuracy,count
0,abstract_algebra,0.500000,20
1,anatomy,0.629630,27
2,astronomy,0.966667,30
3,business_ethics,0.800000,20
4,clinical_knowledge,0.830189,53
5,college_biology,0.862069,29
6,college_chemistry,0.700000,20
7,college_computer_science,0.750000,20
8,college_mathematics,0.600000,20
9,college_medicine,0.800000,35


# Pruning

In [8]:
import copy

def prune_model_layers(model, layers_to_remove):
    """
    Удаляет указанные слои из модели.
    layers_to_remove: список индексов слоев (например, [15, 16, 17, 18])
    """
    # В моделях архитектуры Llama/Qwen слои лежат в model.model.layers
    old_layers = model.model.layers
    new_layers = torch.nn.ModuleList()
    
    for i, layer in enumerate(old_layers):
        if i not in layers_to_remove:
            new_layers.append(layer)
    
    # Заменяем слои в модели
    model.model.layers = new_layers
    # Обновляем конфиг, чтобы модель знала, сколько теперь слоев
    model.config.num_hidden_layers = len(new_layers)
    
    print(f"Layers removed: {layers_to_remove}")
    print(f"New layer count: {len(model.model.layers)}")
    return model

In [9]:
# 1. Загружаем чистый Baseline в FP16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cpu") # На CPU, чтобы не забить GPU

# 2. Удаляем слои (давай начнем аккуратно, удалим 4 слоя из 32)
# Обычно в Qwen3-8B 32 слоя. Попробуем убрать 16, 17, 18, 19.
model = prune_model_layers(model, layers_to_remove=[16, 17, 18, 19])

# 3. Переносим на GPU и квантуем (4-bit, group_size=128)
model = model.to("cuda")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
2025-12-22 21:23:49.786453: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766438630.180564      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766438630.293904      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766438631.348107      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766438631.348145      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766438631.348148      55

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Layers removed: [16, 17, 18, 19]
New layer count: 32


# Quantization

In [8]:
import torch
from hqq.models.hf.base import AutoHQQHFModel
from hqq.core.quantize import BaseQuantizeConfig

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

hqq_config = BaseQuantizeConfig(nbits=NBITS, group_size=128)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    # quantization_config=hqq_config,
    device_map="auto",
    dtype=torch.float16
)

print(f"Quantizing model to {NBITS}-bit...")
AutoHQQHFModel.quantize_model(model, quant_config=hqq_config, compute_dtype=torch.float16, device="cuda:1")

# model.save_pretrained(SAVE_PATH)
AutoHQQHFModel.save_quantized(model, SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model successfully compressed and saved to {SAVE_PATH}")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

2025-12-23 12:57:37.443769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766494657.664777      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766494657.731748      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766494658.261198      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766494658.261224      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766494658.261227      55 computation_placer.cc:177] computation placer alr

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Quantizing model to 4-bit...


100%|██████████| 253/253 [00:40<00:00,  6.21it/s]


Model successfully compressed and saved to qwen3-8b-hqq-4bit


## 4 bit

### Group Size 64

In [10]:
model_hqq_4bit = AutoHQQHFModel.from_quantized(SAVE_PATH)
tokenizer_hqq_4bit = AutoTokenizer.from_pretrained(SAVE_PATH, fix_mistral_regex=True)

100%|██████████| 253/253 [00:00<00:00, 8921.58it/s]


In [11]:
# Считаем параметры
hqq_4bit_params = sum(p.numel() for p in model_hqq_4bit.parameters())
print(f"Quantized parameters: {hqq_4bit_params / 1e9:.2f}B")

Quantized parameters: 4.72B


In [12]:
test_df = load_mmlu(SUBSET_RATIO)

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(


In [13]:
hqq_4bit_metric, hqq_4bit_detailed_metric = evaluate_model(model_hqq_4bit, tokenizer_hqq_4bit, test_df)
print(f"Quantized Metric (MMLU subset): {hqq_4bit_metric:.4f}")
hqq_4bit_detailed_metric

Evaluating: 100%|██████████| 2809/2809 [34:59<00:00,  1.34it/s]

Quantized Metric (MMLU subset): 0.6949


,subject,accuracy,count
0,abstract_algebra,0.400000,20
1,anatomy,0.592593,27
2,astronomy,0.966667,30
3,business_ethics,0.700000,20
4,clinical_knowledge,0.867925,53
5,college_biology,0.862069,29
6,college_chemistry,0.550000,20
7,college_computer_science,0.750000,20
8,college_mathematics,0.450000,20
9,college_medicine,0.742857,35


In [14]:
results = calculate_lab_metrics(ORIG_METRIC, hqq_4bit_metric, None, SAVE_PATH, ORIG_SIZE)

print(f"RESULTS FOR {NBITS}-BIT COMPRESSION:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

RESULTS FOR 4-BIT COMPRESSION:
Original Size (MB): 15622.6315
Compressed Size (MB): 6100.8733
Ratio: 2.5607
Drop: 0.0221
Final Score: 2.5054


### Group Size 128

In [11]:
model_hqq_4bit = AutoHQQHFModel.from_quantized(SAVE_PATH)
tokenizer_hqq_4bit = AutoTokenizer.from_pretrained(SAVE_PATH, fix_mistral_regex=True)

100%|██████████| 253/253 [00:00<00:00, 8339.82it/s]


In [12]:
# Считаем параметры
hqq_4bit_params = sum(p.numel() for p in model_hqq_4bit.parameters())
print(f"Quantized parameters: {hqq_4bit_params / 1e9:.2f}B")

Quantized parameters: 4.72B


In [13]:
test_df = load_mmlu(SUBSET_RATIO)
hqq_4bit_metric, hqq_4bit_detailed_metric = evaluate_model(model_hqq_4bit, tokenizer_hqq_4bit, test_df)
print(f"Quantized Metric (MMLU subset): {hqq_4bit_metric:.4f}")

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(
Evaluating: 100%|██████████| 2809/2809 [34:26<00:00,  1.36it/s]

Quantized Metric (MMLU subset): 0.6913


In [14]:
hqq_4bit_detailed_metric

,subject,accuracy,count
0,abstract_algebra,0.500000,20
1,anatomy,0.592593,27
2,astronomy,0.966667,30
3,business_ethics,0.800000,20
4,clinical_knowledge,0.867925,53
5,college_biology,0.896552,29
6,college_chemistry,0.650000,20
7,college_computer_science,0.700000,20
8,college_mathematics,0.450000,20
9,college_medicine,0.742857,35


In [15]:
results = calculate_lab_metrics(ORIG_METRIC, hqq_4bit_metric, None, SAVE_PATH, ORIG_SIZE)

print(f"RESULTS FOR {NBITS}-BIT COMPRESSION:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

RESULTS FOR 4-BIT COMPRESSION:
Original Size (MB): 15622.6315
Compressed Size (MB): 5893.8723
Ratio: 2.6507
Drop: 0.0271
Final Score: 2.5807


### `axis=0`

In [10]:
model_hqq_4bit = AutoHQQHFModel.from_quantized(SAVE_PATH)
tokenizer_hqq_4bit = AutoTokenizer.from_pretrained(SAVE_PATH, fix_mistral_regex=True)

100%|██████████| 253/253 [00:00<00:00, 8933.59it/s]


In [11]:
# Считаем параметры
hqq_4bit_params = sum(p.numel() for p in model_hqq_4bit.parameters())
print(f"Quantized parameters: {hqq_4bit_params / 1e9:.2f}B")

Quantized parameters: 4.72B


In [12]:
test_df = load_mmlu(SUBSET_RATIO)
hqq_4bit_metric, hqq_4bit_detailed_metric = evaluate_model(model_hqq_4bit, tokenizer_hqq_4bit, test_df)
print(f"Quantized Metric (MMLU subset): {hqq_4bit_metric:.4f}")

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(
Evaluating: 100%|██████████| 2809/2809 [39:30<00:00,  1.19it/s]

Quantized Metric (MMLU subset): 0.6828


In [13]:
hqq_4bit_detailed_metric

,subject,accuracy,count
0,abstract_algebra,0.550000,20
1,anatomy,0.629630,27
2,astronomy,0.900000,30
3,business_ethics,0.750000,20
4,clinical_knowledge,0.811321,53
5,college_biology,0.931034,29
6,college_chemistry,0.650000,20
7,college_computer_science,0.650000,20
8,college_mathematics,0.550000,20
9,college_medicine,0.742857,35


In [14]:
results = calculate_lab_metrics(ORIG_METRIC, hqq_4bit_metric, None, SAVE_PATH, ORIG_SIZE)

print(f"RESULTS FOR {NBITS}-BIT COMPRESSION:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

RESULTS FOR 4-BIT COMPRESSION:
Original Size (MB): 15622.6315
Compressed Size (MB): 5893.8740
Ratio: 2.6507
Drop: 0.0391
Final Score: 2.5509


### Pruned

In [16]:
model_hqq_4bit = AutoHQQHFModel.from_quantized(SAVE_PATH)
tokenizer_hqq_4bit = AutoTokenizer.from_pretrained(SAVE_PATH, fix_mistral_regex=True)

100%|██████████| 225/225 [00:00<00:00, 6780.95it/s]


In [17]:
# Считаем параметры
hqq_4bit_params = sum(p.numel() for p in model_hqq_4bit.parameters())
print(f"Quantized parameters: {hqq_4bit_params / 1e9:.2f}B")

Quantized parameters: 4.33B


In [18]:
test_df = load_mmlu(SUBSET_RATIO)
hqq_4bit_metric, hqq_4bit_detailed_metric = evaluate_model(model_hqq_4bit, tokenizer_hqq_4bit, test_df)
print(f"Quantized Metric (MMLU subset): {hqq_4bit_metric:.4f}")

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(
Evaluating: 100%|██████████| 2809/2809 [32:10<00:00,  1.45it/s]

Quantized Metric (MMLU subset): 0.4432


In [19]:
hqq_4bit_detailed_metric

,subject,accuracy,count
0,abstract_algebra,0.200000,20
1,anatomy,0.333333,27
2,astronomy,0.533333,30
3,business_ethics,0.300000,20
4,clinical_knowledge,0.698113,53
5,college_biology,0.275862,29
6,college_chemistry,0.350000,20
7,college_computer_science,0.400000,20
8,college_mathematics,0.200000,20
9,college_medicine,0.514286,35


In [20]:
results = calculate_lab_metrics(ORIG_METRIC, hqq_4bit_metric, None, SAVE_PATH, ORIG_SIZE)

print(f"RESULTS FOR {NBITS}-BIT COMPRESSION:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

RESULTS FOR 4-BIT COMPRESSION:
Original Size (MB): 15622.6315
Compressed Size (MB): 5502.7748
Ratio: 2.8390
Drop: 0.3763
Final Score: 2.0628


## 3 bit

In [12]:
model_hqq_3bit = AutoHQQHFModel.from_quantized(SAVE_PATH)
tokenizer_hqq_3bit = AutoTokenizer.from_pretrained(SAVE_PATH, fix_mistral_regex=True)

100%|██████████| 253/253 [00:00<00:00, 8643.68it/s]


In [13]:
# Считаем параметры
hqq_3bit_params = sum(p.numel() for p in model_hqq_3bit.parameters())
print(f"Quantized parameters: {hqq_3bit_params / 1e9:.2f}B")

Quantized parameters: 1.94B


In [14]:
test_df = load_mmlu(SUBSET_RATIO)

hqq_3bit_metric, hqq_3bit_detailed_metric = evaluate_model(model_hqq_3bit, tokenizer_hqq_3bit, test_df)
print(f"Quantized Metric (MMLU subset, 3 bit): {hqq_3bit_metric:.4f}")

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(
Evaluating: 100%|██████████| 2809/2809 [48:30<00:00,  1.04s/it]

Quantized Metric (MMLU subset, 3 bit): 0.5554


In [15]:
results = calculate_lab_metrics(ORIG_METRIC, hqq_3bit_metric, None, SAVE_PATH, ORIG_SIZE)

print(f"RESULTS FOR {NBITS}-BIT COMPRESSION:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

RESULTS FOR 3-BIT COMPRESSION:
Original Size (MB): 15622.6315
Compressed Size (MB): 5438.5101
Ratio: 2.8726
Drop: 0.2185
Final Score: 2.3575


### Other Hyperparams

In [10]:
model_hqq_3bit = AutoHQQHFModel.from_quantized(SAVE_PATH)
tokenizer_hqq_3bit = AutoTokenizer.from_pretrained(SAVE_PATH, fix_mistral_regex=True)

100%|██████████| 253/253 [00:00<00:00, 7238.76it/s]


In [11]:
# Считаем параметры
hqq_3bit_params = sum(p.numel() for p in model_hqq_3bit.parameters())
print(f"Quantized parameters: {hqq_3bit_params / 1e9:.2f}B")

Quantized parameters: 1.94B


In [12]:
test_df = load_mmlu(SUBSET_RATIO)

hqq_3bit_metric, hqq_3bit_detailed_metric = evaluate_model(model_hqq_3bit, tokenizer_hqq_3bit, test_df)
print(f"Quantized Metric (MMLU subset, 3 bit, group size 32): {hqq_3bit_metric:.4f}")

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(
Evaluating: 100%|██████████| 2809/2809 [49:40<00:00,  1.06s/it]

Quantized Metric (MMLU subset, 3 bit, group size 32): 0.5899


In [13]:
results = calculate_lab_metrics(ORIG_METRIC, hqq_3bit_metric, None, SAVE_PATH, ORIG_SIZE)

print(f"RESULTS FOR {NBITS}-BIT COMPRESSION WITH GROUP SIZE 32:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

RESULTS FOR 3-BIT COMPRESSION WITH GROUP SIZE 32:
Original Size (MB): 15622.6315
Compressed Size (MB): 5852.4893
Ratio: 2.6694
Drop: 0.1699
Final Score: 2.2818


# Save Best Model

In [16]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_lab_token")

In [17]:
import os
from huggingface_hub import HfApi

# ПАРАМЕТРЫ - ЗАПОЛНИТЕ ИХ
USERNAME = "Neuro-Poplar" 
REPO_NAME = f"qwen3-8b-hqq-{NBITS}bit"

repo_id = f"{USERNAME}/{REPO_NAME}"

# Инициализируем API с токеном напрямую
api = HfApi(token=HF_TOKEN)

print(f"Создание репозитория {repo_id}...")
try:
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print("Репозиторий создан.")
except Exception as e:
    print(f"Ошибка при создании: {e}")

print("Загрузка.")
try:
    api.upload_folder(
        folder_path=SAVE_PATH,
        repo_id=repo_id,
        repo_type="model"
    )
    print(f"\nЗагрузка завершена. Ссылка: https://huggingface.co/{repo_id}")
except Exception as e:
    print(f"Ошибка при загрузке: {e}")

Создание репозитория Neuro-Poplar/qwen3-8b-hqq-4bit...
Репозиторий создан.
Загрузка.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Загрузка завершена. Ссылка: https://huggingface.co/Neuro-Poplar/qwen3-8b-hqq-4bit
